# 04 (v1.1). RAG 의료 QA 챗봇 데모

v1.1 의료 DB(문단 142,605개)에 **LLM 답변 생성**을 붙여서, 실제로 물어보면 근거와 함께 답하는 챗봇까지 만듭니다.

```
질문 ─▶ bge 검색 100개 ─▶ MiniLM 재정렬 ─▶ 상위 5개 문단 ─▶ Qwen2.5-7B ─▶ 답변 + 출처 [번호]
```

| 단계 | 내용 |
|---|---|
| 3. 검색기 | 03과 같은 방식. 재정렬은 MiniLM-L-6-v2 (리랭커 5개 비교에서 MRR 가장 높고 작음), FP16 |
| 4. LLM | `Qwen/Qwen2.5-7B-Instruct`, 문서 안의 내용으로만 답하고 문장마다 [번호] 출처 |
| 5. 거절 기준 | 재정렬 최고 점수가 낮으면 답하지 않음. 기준은 **검증(val) 질문으로만** 정함 |
| 6. 평가 | 시험 건강 질문 100개 + 분야 밖 50개. **LLM 단독(B0)** vs **RAG 기본** vs **RAG 개선**(중복 제거 + 거절 기준) |
| 7. 챗봇 | Gradio 공유 링크. 오른쪽에 근거 문단과 출처 등급 표시. 대화는 드라이브에 기록 |
| 8. 종료 | 드라이브 저장 마무리 + 런타임 종료 |

**런타임: A100.** 위에서부터 Run all 하세요. 7번 챗봇 셀은 계속 실행 상태로 링크를 띄워 둡니다.
챗봇을 다 쓰고 그 셀의 **■(중지)** 를 누르면 챗봇이 닫히고 8번 종료 셀이 이어서 실행됩니다.

결과는 드라이브 `rag_msmarco/v11_test/rag/`에 저장됩니다.

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu gradio

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/v11_test'    # T2 (v1.1) 결과 폴더
OUT_DIR  = f'{SAVE_DIR}/rag'
os.makedirs(OUT_DIR, exist_ok=True)
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)

## 2. 설정

`N_TEST`를 줄이면 평가가 빨라집니다 (예: 50). 시간이 밀리면 여기부터 줄이세요.

In [ ]:
import re, json, time, random
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
TOP_N      = 5          # LLM에 넣을 문단 수
N_TEST     = 100        # 평가할 건강 질문 수 (분야 밖 50개는 전부 사용)
GEN_BATCH  = 16         # 평가 때 한 번에 생성할 질문 수
MAX_NEW    = 256        # 답변 최대 토큰

EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. 검색기 (bge → MiniLM 재정렬)

- `dedup=True`: 본문이 같은 문단은 하나만 남깁니다 (URL만 다르고 내용이 같은 문단이 상위 자리를 차지하는 문제).
- 출처 등급: 공공기관 · 병원/학회 · 상업 매체 · 커뮤니티/생활 · 사전 · 기타

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'
print(f'문단 {len(corpus):,} / 평가 질문 {len(eval_set)} {Counter(q["domain"] for q in eval_set)}')

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

TIERS = {
    '공공기관':   ['medlineplus.gov', 'cdc.gov', 'nih.gov', 'nhs.uk', 'who.int', 'fda.gov', 'cancer.gov'],
    '병원·학회':  ['mayoclinic.org', 'clevelandclinic.org', 'hopkinsmedicine.org', 'aafp.org', 'merckmanuals.com',
                  'kidshealth.org', 'health.harvard.edu', 'heart.org', 'cancer.org', 'familydoctor.org', 'uptodate.com'],
    '상업 매체':  ['webmd.com', 'healthline.com', 'verywell.com', 'medicinenet.com', 'everydayhealth.com', 'drugs.com',
                  'rxlist.com', 'emedicinehealth.com', 'medscape.com', 'medicalnewstoday.com', 'rightdiagnosis.com',
                  'patient.info', 'labtestsonline.org'],
    '커뮤니티·생활': ['healthtap.com', 'sharecare.com', 'livestrong.com', 'babycenter.com', 'psychologytoday.com'],
    '사전':       ['thefreedictionary.com'],
}

def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h

def tier_of(site):
    for tier, sites in TIERS.items():
        if any(site == s or site.endswith('.' + s) for s in sites):
            return tier
    return '기타'

norm_text = lambda t: ' '.join(t.lower().split())

def encode(strings):
    return emb_model.encode([QUERY_PREFIX + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def retrieve_many(queries, dedup=True, top_n=TOP_N):
    # 질문 여러 개 → 질문마다 [{pid, score, site, tier, url, text}, ...] 상위 top_n개
    _, ids = index.search(encode(queries), K)
    out = []
    for q, cands in zip(queries, ids.tolist()):
        scores = reranker.predict([(q, texts[p]) for p in cands], batch_size=128, show_progress_bar=False)
        docs, seen = [], set()
        for i in np.argsort(-scores):
            p = cands[i]
            key = norm_text(texts[p])
            if dedup and key in seen:
                continue
            seen.add(key)
            site = site_of(corpus[p]['url'])
            docs.append({'pid': int(p), 'score': float(scores[i]), 'site': site, 'tier': tier_of(site),
                         'url': corpus[p]['url'], 'text': texts[p]})
            if len(docs) == top_n:
                break
        out.append(docs)
    return out

retrieve = lambda q, **kw: retrieve_many([q], **kw)[0]

In [ ]:
# 검색 점검: 질문 3개
for q in ['what causes boils to suddenly appear', 'how long does the flu last', 'what is the normal blood sugar level']:
    t = time.time()
    docs = retrieve(q)
    print(f'\n질문: {q}  ({(time.time() - t) * 1000:.0f} ms)')
    for i, d in enumerate(docs, 1):
        print(f'  [{i}] {d["score"]:6.2f}  {d["tier"]:7s} {d["site"]:35s} {d["text"][:80]}')

## 4. LLM 답변 생성 (Qwen2.5-7B-Instruct)

- **RAG**: 문서 [1]~[5]에 있는 내용으로만 답하고, 문장마다 [번호]. 문서에 답이 없거나 의료 질문이 아니면 정해진 거절 문장.
- **B0 (LLM 단독)**: 문서 없이 모델 지식만으로 답함. 비교 기준.
- 데이터가 영어라서 질문도 **영어**로 해야 검색이 잘 됩니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.bfloat16, device_map='cuda')
llm.eval()
print('LLM 불러옴:', LLM_NAME, f'/ GPU 메모리 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

REFUSAL = "Sorry, I can't answer that from the medical documents I have."

SYS_RAG = f'''You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences.'''

SYS_B0 = f'''You are a medical information assistant.
Answer the question from your own knowledge in 2-4 short sentences.
If the question is not about health or medicine, reply with exactly this sentence: "{REFUSAL}"'''

def rag_messages(q, docs):
    ctx = '\n'.join(f'[{i}] ({d["site"]}) {d["text"]}' for i, d in enumerate(docs, 1))
    return [{'role': 'system', 'content': SYS_RAG},
            {'role': 'user', 'content': f'Documents:\n{ctx}\n\nQuestion: {q}'}]

def b0_messages(q):
    return [{'role': 'system', 'content': SYS_B0}, {'role': 'user', 'content': q}]

@torch.inference_mode()
def generate(msgs_list, bs=GEN_BATCH):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

In [ ]:
# 생성 점검: 건강 질문 3개 + 분야 밖 2개
health = [q for q in eval_set if q['domain'] == 'health']
others = [q for q in eval_set if q['domain'] != 'health']
for q in health[:3] + others[:2]:
    t = time.time()
    docs = retrieve(q['query'])
    ans = generate([rag_messages(q['query'], docs)])[0]
    print(f"\n[{q['domain']}] {q['query']}   ({time.time() - t:.1f}초)")
    print('  RAG :', ans)
    print('  정답:', q['answers'][0][:200] if q.get('answers') else '-')

## 5. 거절 기준 정하기 (검증 질문으로만)

재정렬 최고 점수가 너무 낮으면 "관련 문서가 없다"고 보고 LLM을 부르지 않고 거절합니다.
기준값은 **검증(val) 200개**(모두 답 있는 건강 질문)의 최고 점수 하위 5% 지점으로 정합니다.
즉 답할 수 있는 질문을 잘못 거절하는 비율을 약 5%로 맞추는 기준입니다. 시험 질문은 보지 않습니다.

In [ ]:
val_set = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
val_top = [docs[0]['score'] for docs in retrieve_many([q['query'] for q in val_set])]
THRESH = float(np.percentile(val_top, 5))
print(f'검증 {len(val_set)}개 최고 점수: 중앙값 {np.median(val_top):.2f} / 하위 5% = 거절 기준 {THRESH:.2f}')

## 6. 평가: LLM 단독 vs RAG 기본 vs RAG 개선

| 시스템 | 문서 | 중복 제거 | 거절 기준 |
|---|---|---|---|
| B0 | 없음 | - | - |
| RAG 기본 | 상위 5개 | ✗ | ✗ |
| RAG 개선 | 상위 5개 | ✓ | ✓ |

**지표**
- 건강 질문: 정답과 단어 겹침(**F1**, **ROUGE-L**), **출처 표시율**(답변에 [번호]), **잘못 거절**(답 있는데 거절), 근거 R@5(정답 문단이 상위 5개 안에)
- 분야 밖 질문: **거절률** (높을수록 좋음)
- 질문당 생성 시간 (16개씩 묶어서 생성한 평균)

정답 문장에 오타·잘림이 섞여 있어서 F1·ROUGE-L은 **상대 비교용**입니다.

In [ ]:
import string

def toks(s):
    s = s.lower()
    s = re.sub(r'\[\d+\]', ' ', s)                          # 출처 번호는 채점에서 제외
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def rouge_l(pred, ref):
    p, r = toks(pred), toks(ref)
    if not p or not r:
        return 0.0
    dp = [0] * (len(r) + 1)
    for a in p:
        prev = 0
        for j, b in enumerate(r, 1):
            cur = dp[j]
            dp[j] = prev + 1 if a == b else max(dp[j], dp[j - 1])
            prev = cur
    lcs = dp[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

best = lambda fn, pred, refs: max(fn(pred, r) for r in refs)
has_cite = lambda a: bool(re.search(r'\[\d\]', a))

In [ ]:
test_h = random.Random(SEED).sample([q for q in eval_set if q['domain'] == 'health'], N_TEST)
test_o = [q for q in eval_set if q['domain'] != 'health']
qs = test_h + test_o
queries = [q['query'] for q in qs]
print(f'평가: 건강 {len(test_h)}개 + 분야 밖 {len(test_o)}개')

runs = {}

t = time.time()
runs['B0'] = {'answers': generate([b0_messages(q) for q in queries]), 'docs': [[] for _ in qs]}
runs['B0']['sec'] = (time.time() - t) / len(qs)
print(f"B0 끝 ({runs['B0']['sec']:.2f}초/질문)")

for name, dedup, use_thresh in [('RAG 기본', False, False), ('RAG 개선', True, True)]:
    t = time.time()
    docs_all = retrieve_many(queries, dedup=dedup)
    answers = [None] * len(qs)
    todo = [i for i, d in enumerate(docs_all) if not (use_thresh and d[0]['score'] < THRESH)]
    for i in set(range(len(qs))) - set(todo):
        answers[i] = REFUSAL                                  # 관련 문서 없음 → LLM 안 부르고 거절
    for i, a in zip(todo, generate([rag_messages(queries[i], docs_all[i]) for i in todo])):
        answers[i] = a
    runs[name] = {'answers': answers, 'docs': docs_all, 'sec': (time.time() - t) / len(qs),
                  'thresh_refused': len(qs) - len(todo)}
    print(f"{name} 끝 ({runs[name]['sec']:.2f}초/질문, 점수 기준 거절 {runs[name]['thresh_refused']}개)")

In [ ]:
import pandas as pd

nh = len(test_h)
rows = {}
for name, r in runs.items():
    A, D = r['answers'], r['docs']
    h = range(nh)
    rows[name] = {
        'F1':          np.mean([best(f1, A[i], qs[i]['answers']) for i in h]),
        'ROUGE-L':     np.mean([best(rouge_l, A[i], qs[i]['answers']) for i in h]),
        '출처 표시율':   np.mean([has_cite(A[i]) for i in h if not is_refusal(A[i])]) if D[0] else np.nan,
        '잘못 거절':     np.mean([is_refusal(A[i]) for i in h]),
        '근거 R@5':     np.mean([bool(set(qs[i]['gold_pids']) & {d['pid'] for d in D[i]}) for i in h]) if D[0] else np.nan,
        '분야 밖 거절률': np.mean([is_refusal(A[i]) for i in range(nh, len(qs))]),
        '초/질문':      r['sec'],
    }
table = pd.DataFrame(rows).T.round(3)
display(table)

summary = '\n'.join([
    f"# 04 RAG 챗봇 평가 ({time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))} KST)", '',
    f'- DB: 문단 {len(corpus):,}개 (v1.1) / 검색: bge → {RERANKER.split("/")[-1]} 상위 {TOP_N}개 / LLM: {LLM_NAME}',
    f'- 평가: 건강 {nh}개 + 분야 밖 {len(test_o)}개 / 거절 기준(검증 하위 5%) = {THRESH:.2f}',
    '- F1·ROUGE-L은 정답 문장과의 단어 겹침 (정답에 오타·잘림이 있어 상대 비교용)', '',
    table.to_markdown(),
])
with open(f'{OUT_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)
with open(f'{OUT_DIR}/results.json', 'w') as f:
    json.dump({'thresh': THRESH, 'n_health': nh, 'n_other': len(test_o), 'table': table.to_dict('index')}, f, indent=2,
              ensure_ascii=False)
save_jsonl([{'qid': q['qid'], 'domain': q['domain'], 'query': q['query'], 'gold_answer': (q.get('answers') or [''])[0],
             **{name: r['answers'][i] for name, r in runs.items()},
             'rag_docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in runs['RAG 개선']['docs'][i]]}
            for i, q in enumerate(qs)], f'{OUT_DIR}/eval_answers.jsonl')
print('저장:', OUT_DIR)

In [ ]:
# 답변 비교 예시 (번호를 바꿔 보세요. 0~99 건강, 100~ 분야 밖)
for i in [0, 1, nh]:
    q = qs[i]
    print(f"\n[{q['domain']}] {q['query']}")
    print('  정답    :', (q.get('answers') or ['-'])[0][:250])
    for name in runs:
        print(f'  {name:7s}:', runs[name]['answers'][i])

## 7. 챗봇 (Gradio)

아래 셀을 실행하면 `https://xxxx.gradio.live` 공유 링크가 나옵니다. 폰이나 팀원 컴퓨터에서도 열립니다.
- 검색은 **RAG 개선** 설정(중복 제거 + 거절 기준)을 씁니다.
- 질문과 답은 드라이브 `rag/chat_log.jsonl`에 쌓입니다.
- **다 쓰면 이 셀의 ■(중지)를 누르세요.** 챗봇이 닫히고 8번 종료 셀이 이어서 실행됩니다.

In [ ]:
import gradio as gr

EXAMPLES = ['What are the symptoms and causes of conjunctivitis?',
            'How long does the flu last?',
            'What is a normal blood pressure for adults?',
            'Who won the world cup in 2014?']

def answer_one(q):
    t = time.time()
    docs = retrieve(q, dedup=True)
    if docs[0]['score'] < THRESH:
        ans, used = REFUSAL, False
    else:
        ans, used = generate([rag_messages(q, docs)])[0], True
    return ans, docs, used, time.time() - t

def sources_md(docs, used, sec):
    head = f'**근거 문단 상위 {len(docs)}개** · {sec:.1f}초'
    if not used:
        head += f'\n\n⚠️ 재정렬 최고 점수 {docs[0]["score"]:.2f} < 기준 {THRESH:.2f} → 관련 문서가 없다고 보고 답하지 않았어요.'
    parts = [head]
    for i, d in enumerate(docs, 1):
        parts.append(f'**[{i}] {d["tier"]}** · `{d["site"]}` · 점수 {d["score"]:.2f}\n\n> {d["text"]}\n\n[{d["url"]}]({d["url"]})')
    return '\n\n---\n\n'.join(parts)

def chat(message, history):
    message = (message or '').strip()
    if not message:
        return '', history, ''
    ans, docs, used, sec = answer_one(message)
    with open(f'{OUT_DIR}/chat_log.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps({'time': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time() + 9 * 3600)),
                            'question': message, 'answer': ans, 'answered': used,
                            'docs': [{k: d[k] for k in ('pid', 'score', 'site', 'tier')} for d in docs]},
                           ensure_ascii=False) + '\n')
    history = history + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': ans}]
    return '', history, sources_md(docs, used, sec)

with gr.Blocks(title='의료 QA 챗봇 (RAG 데모)') as demo:
    gr.Markdown('## 의료 QA 챗봇 · RAG 데모\nMS MARCO 건강 문서 14만 개에서 근거를 찾아 답합니다. **영어로 질문하세요.**')
    with gr.Row():
        with gr.Column(scale=3):
            try:
                bot = gr.Chatbot(type='messages', height=480, label='대화')   # Gradio 5
            except TypeError:
                bot = gr.Chatbot(height=480, label='대화')                    # Gradio 6부터 messages 형식이 기본
            box = gr.Textbox(placeholder='e.g. What causes migraines?', label='질문', autofocus=True)
            gr.Examples(EXAMPLES, inputs=box)
            clear = gr.Button('대화 지우기')
        with gr.Column(scale=2):
            src = gr.Markdown('질문하면 근거 문단이 여기에 나옵니다.')
    gr.Markdown('<small>⚠️ 데모용입니다. 의료 조언이 아니며, 문서는 2016년 무렵 자료라 최신 정보와 다를 수 있습니다.</small>')
    box.submit(chat, [box, bot], [box, bot, src])
    clear.click(lambda: ([], '질문하면 근거 문단이 여기에 나옵니다.'), None, [bot, src])

demo.queue().launch(share=True, prevent_thread_lock=True)
try:
    while True:                       # ■(중지)를 누를 때까지 챗봇을 열어 둠
        time.sleep(1)
except KeyboardInterrupt:
    demo.close()
    print('챗봇을 닫았습니다. 대화 기록:', f'{OUT_DIR}/chat_log.jsonl')

## 8. 드라이브 저장 마무리 + 런타임 종료

결과 파일은 앞 셀들에서 드라이브(`v11_test/rag/`)에 저장됐습니다. 이 셀은 저장을 마무리하고 Colab 런타임을 끕니다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # 결과 파일 드라이브 저장 마무리
runtime.unassign()          # 런타임 종료